In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import MDS, TSNE
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, accuracy_score
from matplotlib.lines import Line2D
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression

In [ ]:
X = np.load("data/p2_unsupervised/X.npy")
log_X = np.log2(X + 1)

In [ ]:
log_X.shape

2169 cells x 45768 genes

In [ ]:
# largest entry in the first column of processed data:
np.max(log_X[:, 0])

# Visualization

In [ ]:
pca = PCA(n_components=50)
z = pca.fit_transform(log_X)
plt.scatter(z[:,0],z[:,1])
plt.title("Data processed with PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

In [ ]:
mds=MDS(n_components=2).fit_transform(z)
plt.scatter(mds[:,0], mds[:,1])
plt.title("Data processed with MDS")
plt.show()

In [ ]:
z_tsne = TSNE(n_components=2, perplexity=40, random_state=0).fit_transform(z)

In [ ]:
# silhouette scores
major_k_values = range(2, 9)
major_silhouettes = []

for k in major_k_values:
    model = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = model.fit_predict(z)
    score = silhouette_score(z, labels)
    major_silhouettes.append(score)

plt.plot(major_k_values, major_silhouettes, marker="o")
plt.xlabel("Number of major clusters")
plt.ylabel("Silhouette score")
plt.title("Choosing the number of major clusters")
plt.show()

In [ ]:
hc3 = AgglomerativeClustering(n_clusters=3, linkage="ward")
hc_labels = hc3.fit_predict(z)

In [ ]:
subtype_labels = np.full(len(z), -1)
subtype_results = {}

major_ids = np.sort(np.unique(hc_labels))
fig, axes_subtypes = plt.subplots(1, len(major_ids), figsize=(18, 5), sharey=True)

for major_id in np.unique(hc_labels):
    idx = np.where(hc_labels == major_id)[0]
    z_major = z[idx]

    max_k = min(25, len(idx) - 1)
    candidate_k = range(2, max_k + 1)
    scores = []

    for k in candidate_k:
        model = AgglomerativeClustering(n_clusters=k, linkage="ward")
        labels = model.fit_predict(z_major)
        score = silhouette_score(z_major, labels)
        scores.append(score)

    best_k = list(candidate_k)[np.argmax(scores)]
    subtype_results[major_id] = {"candidate_k": list(candidate_k), "silhouette_scores": scores, "chosen_k": best_k}

    final_model = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
    subtype_labels[idx] = final_model.fit_predict(z_major)

    ax = axes_subtypes[major_id]
    ax.plot(candidate_k, scores, marker="o")
    ax.axvline(best_k, color="black", linestyle="--", alpha=0.5)
    best_score = scores[list(candidate_k).index(best_k)]
    
    ax.text(best_k, 0.02, f"k = {best_k}", transform=ax.get_xaxis_transform(), ha="center",
        va="bottom", fontsize=9, weight="bold")
    ax.set_xlabel(f"Number of subtypes within major cluster {major_id + 1}")
    ax.grid(alpha=0.2)

axes_subtypes[0].set_ylabel("Silhouette score")
plt.tight_layout()
plt.show()

In [ ]:
major_names = {0: "M1", 1: "M2", 2: "M3"}

# Based on silhouette score plots
chosen_subtypes = {0: 11, 1: 4, 2: 6}

subtype_labels = np.full(len(z), -1)
subtype_to_major = {}

for major_id, n_subtypes in chosen_subtypes.items():

    # Cells belonging to this major cluster
    cell_indices = np.where(hc_labels == major_id)[0]

    z_major = z[cell_indices]

    subtype_model = AgglomerativeClustering(n_clusters=n_subtypes, linkage="ward")

    local_subtype_labels = subtype_model.fit_predict(z_major)

    subtype_labels[cell_indices] = local_subtype_labels

    for subtype_id in range(n_subtypes):
        subtype_to_major[f"M{major_id + 1}-S{subtype_id + 1}"] = major_names[major_id]

# labels
subtype_plot_labels = np.array([f"M{major_id + 1}-S{subtype_id + 1}" for major_id, subtype_id in zip(hc_labels, subtype_labels)])

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# major class plot
major_palette = sns.color_palette("Set2", n_colors=3)
major_colors = dict(enumerate(major_palette))

for major_id in sorted(major_names):
    idx = hc_labels == major_id

    axes[0].scatter(z_tsne[idx, 0], z_tsne[idx, 1], s=8, alpha=0.7, color=major_colors[major_id])

axes[0].set_title("Hierarchical clustering: three major cell classes")
axes[0].set_xlabel("t-SNE 1")
axes[0].set_ylabel("t-SNE 2")
major_legend_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=major_colors[major_id], markersize=7, label=major_names[major_id])
    for major_id in sorted(major_names)]
axes[0].legend(handles=major_legend_handles)
axes[0].grid(alpha=0.2)

# subtype plot
all_subtype_names = []

for major_id in sorted(chosen_subtypes):
    for subtype_id in range(chosen_subtypes[major_id]):
        all_subtype_names.append(f"M{major_id + 1}-S{subtype_id + 1}")

colors = sns.color_palette("husl", n_colors=len(all_subtype_names))
subtype_color_map = dict(zip(all_subtype_names, colors))

for subtype_name in all_subtype_names:
    idx = subtype_plot_labels == subtype_name
    axes[1].scatter(z_tsne[idx, 0], z_tsne[idx, 1], s=8, alpha=0.7, color=subtype_color_map[subtype_name])

# Legend for subtype plot
legend_handles = []

for subtype_name in all_subtype_names:
    major_id = int(subtype_name.split("-")[0][1:]) - 1
    legend_handles.append(Line2D([0],[0], marker="o", color="w", markerfacecolor=subtype_color_map[subtype_name],
            markersize=7, label=subtype_name))

axes[1].legend(handles=legend_handles, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8.5)

axes[1].set_title("Hierarchical clustering: subtypes within major classes")
axes[1].set_xlabel("t-SNE 1")
axes[1].set_ylabel("t-SNE 2")
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

# Logistic regression model - train & validate

In [ ]:
label_encoder = LabelEncoder()

y = label_encoder.fit_transform(subtype_plot_labels)

print("Number of classes:", len(label_encoder.classes_))
print("Classes:", label_encoder.classes_)

In [ ]:
X_model = log_X

In [ ]:
# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_model, y, test_size=0.2, random_state=0, stratify=y)

In [ ]:
# Tune L1 regularization using cross-validation
pipeline = Pipeline([("scale", StandardScaler()),
    ("logistic", LogisticRegression(penalty="l1", solver="liblinear", multi_class="ovr", max_iter=2000))])

In [ ]:
# tune C
C_values = np.logspace(-3, 2, 6)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

grid = GridSearchCV(
    estimator=pipeline,
    param_grid={"logistic__C": C_values},
    scoring="accuracy",
    cv=cv,
    refit=True,
    n_jobs=1,
    pre_dispatch=1,
    return_train_score=True
)

grid.fit(X_train, y_train)

In [ ]:
print("Best C:", grid.best_params_["logistic__C"])
print("Best cross-validation score:", grid.best_score_)

In [ ]:
cv_results = grid.cv_results_

plt.semilogx(C_values, cv_results["mean_test_score"], marker="o")

plt.fill_between(C_values, cv_results["mean_test_score"] - cv_results["std_test_score"],
                 cv_results["mean_test_score"] + cv_results["std_test_score"], alpha=0.2)

plt.xlabel("C")
plt.ylabel("Mean cross-validation accuracy")
plt.title("Selecting the regularization parameter")
plt.grid(alpha=0.2)
plt.show()

In [ ]:
# Evaluate on the validation set
y_val_pred = grid.predict(X_val)

validation_accuracy = accuracy_score(y_val, y_val_pred)

print("Validation accuracy:", validation_accuracy)

In [ ]:
best_pipeline = grid.best_estimator_

best_logistic = best_pipeline.named_steps["logistic"]

In [ ]:
# (number of subtypes, number of genes)
best_logistic.coef_.shape

In [ ]:
gene_importance = np.max(np.abs(best_logistic.coef_), axis=0)

In [ ]:
top_100_indices = np.argsort(gene_importance)[-100:][::-1]

print("Top 100 gene indices:")
print(top_100_indices)

# Logistic regression model - evaluate

In [ ]:
X_eval_train = np.load("data/p2_evaluation/X_train.npy")

X_eval_test = np.load("data/p2_evaluation/X_test.npy")

y_eval_train = np.load("data/p2_evaluation/y_train.npy")

y_eval_test = np.load("data/p2_evaluation/y_test.npy")

X_eval_train = np.log2(X_eval_train + 1)
X_eval_test = np.log2(X_eval_test + 1)

In [ ]:
def train_and_evaluate(X_train_features, X_test_features):

    model_pipeline = Pipeline([
        ("scale", StandardScaler()),
        ("logistic", LogisticRegression(penalty="l1", solver="liblinear", multi_class="ovr", max_iter=2000))])

    model_grid = GridSearchCV(
        estimator=model_pipeline,
        param_grid={
            "logistic__C": C_values
        },
        scoring="accuracy",
        cv=StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=0
        ),
        refit=True,
        n_jobs=1,
        pre_dispatch=1
    )

    model_grid.fit(X_train_features, y_eval_train)

    predictions = model_grid.predict(X_test_features)

    test_score = accuracy_score(y_eval_test, predictions)

    return model_grid, test_score

In [ ]:
# evaluate 100 selected genes
X_selected_train = X_eval_train[:, top_100_indices]
X_selected_test = X_eval_test[:, top_100_indices]

selected_model, selected_score = train_and_evaluate(X_selected_train, X_selected_test)

print("Selected feature test accuracy:", selected_score)
print("Selected feature best C:", selected_model.best_params_["logistic__C"])

In [ ]:
# Use the C selected by the logistic regression model
C_for_comparison = selected_model.best_params_["logistic__C"]
def train_and_evaluate_fixed_C(X_train_features, X_test_features, C_value):
    model = Pipeline([
        ("scale", StandardScaler()),
        ("logistic", LogisticRegression(penalty="l1", solver="liblinear", multi_class="ovr", C=C_value, max_iter=2000,))])

    model.fit(X_train_features, y_eval_train)

    predictions = model.predict(X_test_features)

    score = accuracy_score(y_eval_test, predictions)

    return model, score

In [ ]:
# Select 100 random genes
rng = np.random.default_rng(0)

random_indices = rng.choice(X_eval_train.shape[1], size=100, replace=False)

X_random_train = X_eval_train[:, random_indices]
X_random_test = X_eval_test[:, random_indices]

random_model, random_score = train_and_evaluate_fixed_C(X_random_train, X_random_test, C_for_comparison)

print("Random feature test accuracy:", random_score)

In [ ]:
# select 100 genes with the highest variance
gene_variances = np.var(X_eval_train, axis=0)

high_variance_indices = np.argsort(gene_variances)[-100:][::-1]

X_variance_train = X_eval_train[:, high_variance_indices]
X_variance_test = X_eval_test[:, high_variance_indices]

variance_model, variance_score = train_and_evaluate_fixed_C(X_variance_train, X_variance_test, C_for_comparison)

print("Highest variance feature test accuracy:", variance_score)

In [ ]:
print("Selected logistic features:", selected_score)
print("Random features:", random_score)
print("Highest variance features:", variance_score)

In [ ]:
selected_variances = gene_variances[top_100_indices]

high_variance_variances = gene_variances[high_variance_indices]

plt.hist(selected_variances, bins=20, alpha=0.6, label="Logistic regression selected genes")

plt.hist(high_variance_variances, bins=20, alpha=0.6, label="Highest variance genes")

plt.xlabel("Gene variance")
plt.ylabel("Number of genes")
plt.title("Variance of selected gene sets")
plt.legend()
plt.grid(alpha=0.2)
plt.show()